In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation
from math import cos, sin, pi
from h5_utils import load, save
from quat_utils import quatconj, quatmultiply
import os
import pyvista as pv
import h5py
from scipy.linalg import eig
from rpca import rpca_on_scans

In [17]:
# 1. Define the file path
FILE_PATH = '6_TRAJECTORY_AND_SONAR.h5'

# 2. Load the Trajectory (H)
TRAJ = load(FILE_PATH, 'TRAJ')
H    = TRAJ.H       # Shape: (N, 7) - [x, y, z, qw, qx, qy, qz]
TIME = TRAJ.TIME    # Timestamps for the poses

# 3. Load Vertical Sonar Data (M^v)
M_v_data = load(FILE_PATH, 'M_v')
rho_v      = M_v_data.rho       # Distance/Range measurements
psi_v      = M_v_data.psi       # Rotation angles
pose_idx_v = M_v_data.pose_idx.astype(int)  # The mapping function \sigma^v(i)

# 4. Load Horizontal Sonar Data (M^h)
M_h_data = load(FILE_PATH, 'M_h')
# M_h_data.M is a (3, total_returns) array where:
# Row 0 = rho_h, Row 1 = theta_h (NaNs to be estimated), Row 2 = psi_h
rho_h      = M_h_data.M[0, :]   
theta_h    = M_h_data.M[1, :]   # Currently NaNs
psi_h      = M_h_data.M[2, :]   
pose_idx_h = M_h_data.pose_idx.astype(int)  # The mapping function \sigma^h(i)

Loaded TRAJ from 6_TRAJECTORY_AND_SONAR.h5
Loaded M_v from 6_TRAJECTORY_AND_SONAR.h5
Loaded M_h from 6_TRAJECTORY_AND_SONAR.h5


In [18]:
# ---------------------------------------------------------------------------
# Helpers: SE(3) ↔ matrix conversions
# ---------------------------------------------------------------------------

def pose_to_SE3(pose):
    """
    Convert a 7-vector pose [x, y, z, qw, qx, qy, qz] to a 4×4 SE(3) matrix.

    The notebook stores poses as:
        H = np.hstack((X_IMU_ARRAY, QUAT_IMU_ARRAY))   # shape (N, 7)
        where QUAT_IMU is [qw, qx, qy, qz]  (scalar-first convention).
    """
    t              = pose[:3]          # translation [x, y, z]
    qw, qx, qy, qz = pose[3:]          # quaternion (scalar first)
    R              = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()   # 3×3 # scipy.spatial.transform.Rotation uses scalar-last [qx, qy, qz, qw]

    T         = np.eye(4)
    T[:3, :3] = R
    T[:3,  3] = t
    return T


def SE3_log(T):
    """
    SE(3) logarithm map.

    Given a 4×4 transformation matrix T = [R | t ; 0 | 1],
    returns the 6-vector twist  ξ = [v ; ω]  in  se(3).

    The rotation part ω is the axis-angle vector (‖ω‖ = rotation angle).
    The translation part v satisfies  t = V ω  where V is the left-Jacobian
    of SO(3).

    Reference: Eq. (13) in the paper, plus standard SE(3) Lie theory.
    """
    R = T[:3, :3]
    t = T[:3,  3]

    # --- rotation part: ω (axis-angle) ---
    omega = Rotation.from_matrix(R).as_rotvec()   # 3-vector; ‖ω‖ = angle θ
    theta = np.linalg.norm(omega)

    # --- translation part: v = V⁻¹ t ---
    if theta < 1e-9:
        # First-order approximation when rotation is negligible
        v = t.copy()
    else:
        # Skew-symmetric matrix of ω (not unit vector)
        W = np.array([
            [ 0,        -omega[2],  omega[1]],
            [ omega[2],  0,        -omega[0]],
            [-omega[1],  omega[0],  0       ]
        ])
        # Left Jacobian of SO(3):
        #   V = I  +  (1 - cosθ)/θ²  ·  W  +  (θ - sinθ)/θ³  ·  W²
        V = (np.eye(3)
             + (1 - np.cos(theta)) / theta**2 * W
             + (theta - np.sin(theta)) / theta**3 * (W @ W))

        v = np.linalg.solve(V, t)   # v = V⁻¹ t

    return np.concatenate([v, omega])   # 6-vector [v; ω]


# ---------------------------------------------------------------------------
# Algorithm 1 — Line 1:  s = s(H)
# ---------------------------------------------------------------------------

def compute_curvilinear_abscissa(H):
    """
    Compute the curvilinear abscissa for every pose in the trajectory.

    Implements Equation 12  s(t) = L_{η(0)}^{η(t)}
    via the discrete approximation of Equation 13:

        s_i = Σ_{k=1}^{i}  sqrt( log(T_{k-1}⁻¹ T_k)ᵀ G log(T_{k-1}⁻¹ T_k) )

    where the 6×6 matrix G = diag(I₃, 2·I₃)  (Eq. 10) weights translations
    by 1 and rotations by 2, yielding the norm:

        ‖ξ‖_G  =  sqrt( ‖v‖² + 2‖ω‖² )

    Parameters
    ----------
    H : np.ndarray, shape (N, 7)
        Trajectory array.  Each row is [x, y, z, qw, qx, qy, qz].

    Returns
    -------
    s : np.ndarray, shape (N,)
        Cumulative arc length (curvilinear abscissa) for each pose.
        s[0] = 0 by definition (Eq. 12, length from the start).
    """
    N = H.shape[0]
    s = np.zeros(N)   # s[0] = 0 by definition

    # G is block-diagonal: translation weight = 1, rotation weight = 2
    # ‖ξ‖_G² = ξᵀ G ξ = ‖v‖² + 2‖ω‖²
    # We compute this directly rather than building the 6×6 matrix.

    for i in range(1, N):
        T_prev   = pose_to_SE3(H[i - 1])   # T_{k-1}
        T_curr   = pose_to_SE3(H[i])       # T_k

        # Relative (incremental) transformation: ΔT = T_{k-1}⁻¹ · T_k
        # For SE(3): inverse is  [Rᵀ | -Rᵀt ; 0 | 1]
        T_rel    = np.linalg.inv(T_prev) @ T_curr
        xi       = SE3_log(T_rel)  # SE(3) log map → 6-vector [v ; ω]
        v, omega = xi[:3], xi[3:]
        delta_s  = np.sqrt(np.dot(v, v) + 2.0 * np.dot(omega, omega)) # Weighted norm  sqrt(‖v‖² + 2‖ω‖²)  — direct expansion of ξᵀ G ξ
        s[i]     = s[i - 1] + delta_s # Accumulate  (Eq. 13)

    return s

H is "where the robot is," and s is "how far the robot has traveled along its path."

In [4]:
# --- Algorithm 1, Line 1: Compute curvilinear abscissa for the whole trajectory ---
s_traj = compute_curvilinear_abscissa(H)

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; text-align: center; margin-bottom: 20px; font-weight: bold; max-width: 100%; word-wrap: break-word; hyphens: auto;">Horizontal Sonar Full-Scan Detection & Data Structuring</h3>

<p style="text-align: justify; margin: 20px 0; color: #4a4a4a; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box;">
This section groups the continuous stream of individual sonar pings into <strong>complete 360° rotational scans</strong>. By analyzing the angle of the horizontal Ping360 sonar, the algorithm detects when the beam angle resets (wraps around), marking the boundary of a full physical sweep.
</p>

<div style="background-color: #f4eee1; padding: 15px; border-left: 4px solid #bda881; border-radius: 4px; margin: 20px 0;">
<h4 style="margin-top: 0; color: #5d4e37; font-size: 0.95em; font-weight: bold;">The resulting structure, <code style="background-color: #e6ddc5; padding: 2px 5px; border-radius: 3px;">SCAN_DATA</code>, groups the data into 5 key columns per scan:</h4>
<ol style="margin: 10px 0 0 0; color: #4a4a4a; font-size: 0.9em; padding-left: 20px;">
  <li style="margin-bottom: 8px;"><strong>Scan Cycle:</strong> The sequential index of the full 360° scan.</li>
  <li style="margin-bottom: 8px;"><strong>M<sub>v</sub> (Vertical Sonar):</strong> The time-filtered impact points from the SeaKing profiling sonar that fired <em>during</em> this specific scan's time window.</li>
  <li style="margin-bottom: 8px;"><strong>M<sub>h</sub> (Horizontal Sonar):</strong> The valid impact points from the Ping360 sonar across all angles in this rotation.</li>
  <li style="margin-bottom: 8px;"><strong>H (Trajectory):</strong> The exact kinematic poses <code style="background-color: #e6ddc5; padding: 2px 4px; border-radius: 3px;">[x, y, z, qw, qx, qy, qz]</code> the robot executed while performing this sweep.</li>
  <li style="margin-bottom: 0;"><strong>s (Curvilinear Abscissa):</strong> The exact distance the robot traveled along its path during this sweep.</li>
</ol>
</div>

<p style="text-align: justify; margin: 0; color: #4a4a4a; width: 100%; max-width: 100%; word-wrap: break-word; font-style: italic; box-sizing: border-box;">
This guarantees that for any given 360° sonar image, we have the corresponding 3D timeline of exactly where the robot was matching that exact time frame.
</p>

</div>

In [5]:
PING360_VALID = load('3_VALID_DATA.h5', 'PROFILO_PING360_VALID')
POSE_IDX = load('6_TRAJECTORY_AND_SONAR.h5', 'POSE_IDX')
PING360_POSE_IDX = POSE_IDX.PING360_POSE_IDX.astype(int)

# ---------------------------------------------------------------------------
# Detect full-scan boundaries of the horizontal sonar (Ping360)
# ---------------------------------------------------------------------------
# A new scan starts whenever the angle decreases (wraps around)
ping360_angles = PING360_VALID.ANGLE  # already loaded from Input.ipynb pipeline # already loaded from Input.ipynb pipeline

scan_starts = [0]  # First ping always starts a scan
for i in range(1, len(ping360_angles)):
    if ping360_angles[i] < ping360_angles[i - 1]:  # angle wrapped → new scan
        scan_starts.append(i)
scan_starts.append(len(ping360_angles))  # sentinel for the last scan

n_scans = len(scan_starts) - 1
print(f"Detected {n_scans} full scans")

# ---------------------------------------------------------------------------
# Build the per-scan data structure
# ---------------------------------------------------------------------------
SCAN_DATA = []

for k in range(n_scans):
    i_start = scan_starts[k]
    i_end   = scan_starts[k + 1]   # exclusive

    # 1) Scan number (1-indexed)
    scan_number = k + 1

    # 2) Collect M_v entries whose time falls within this scan's time window
    t_scan_start = PING360_VALID.ABS_TIME[i_start]
    t_scan_end   = PING360_VALID.ABS_TIME[i_end - 1]

    # M_v entries in this time window (use M_v_data arrays)
    mv_mask = (M_v_data.time >= t_scan_start) & (M_v_data.time <= t_scan_end)
    mv_in_scan = {
        'rho':      M_v_data.rho[mv_mask],
        'psi':      M_v_data.psi[mv_mask],
        'pose_idx': M_v_data.pose_idx[mv_mask].astype(int),
        'time':     M_v_data.time[mv_mask]
    }
    # M_h entries in this time window (use M_h_data arrays)
    mh_mask = (M_h_data.time >= t_scan_start) & (M_h_data.time <= t_scan_end)
    mh_in_scan = {
        'M':        M_h_data.M[:, mh_mask],
        'pose_idx': M_h_data.pose_idx[mh_mask].astype(int),
        'time':     M_h_data.time[mh_mask]
}
    # (or filter by time similarly)

    # 4) Trajectory H during this scan:
    #    All pose indices referenced by pings in this scan
    pose_indices = PING360_POSE_IDX[i_start:i_end]
    idx_min = min(pose_indices)
    idx_max = max(pose_indices)
    H_scan = H[idx_min:idx_max + 1, :]   # shape (n_poses_in_scan, 7)

    # 5) Curvilinear abscissa for those poses
    s_scan = s_traj[idx_min:idx_max + 1]  # shape (n_poses_in_scan,)

    scan_entry = {
        'scan_number': scan_number,         # Column 1: scan ID
        'M_v':         mv_in_scan,          # Column 2: vertical sonar impacts
        'M_h':         mh_in_scan,          # Column 3: horizontal sonar impacts
        'H':           H_scan,              # Column 4: trajectory during this scan
        's':           s_scan,              # Column 5: curvilinear abscissa
        'pose_range':  (idx_min, idx_max),  # bonus: which poses this covers
    }
    SCAN_DATA.append(scan_entry)

print(f"Built {len(SCAN_DATA)} scan entries")
print(f"Example — Scan 1: {SCAN_DATA[0]['H'].shape[0]} poses, "
      f"s from {SCAN_DATA[0]['s'][0]:.2f} to {SCAN_DATA[0]['s'][-1]:.2f}")


Loaded PROFILO_PING360_VALID from 3_VALID_DATA.h5
Loaded POSE_IDX from 6_TRAJECTORY_AND_SONAR.h5
Detected 303 full scans
Built 303 scan entries
Example — Scan 1: 492 poses, s from 0.00 to 5.28


<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 15px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h4 style="color: #5d4e37; text-align: center; margin-bottom: 15px; font-weight: bold;">3D Point Cloud from Vertical Sonar — Per-Scan Generation</h4>

<p style="text-align: justify; margin: 10px 0; color: #4a4a4a;">
For each scan, the vertical sonar measurements (M<sub>v</sub>) are transformed from the sensor's body-frame into world coordinates using the robot's trajectory. The quaternion-based coordinate transformation pipeline is:
</p>

<ol style="margin: 10px 0; color: #4a4a4a; padding-left: 20px;">
  <li>Convert sonar angle ψ → body-frame quaternion Q<sub>body</sub></li>
  <li>Apply frame alignment rotations (Z by π/2, Y by π/2)</li>
  <li>Combine: Q<sub>combined</sub> = Q<sub>robot</sub> ⊗ Q<sub>body</sub></li>
  <li>Rotate distance: V = Q<sub>combined</sub> ⊗ [0,0,0,ρ] ⊗ Q<sub>combined</sub>*</li>
  <li>World point: P<sub>world</sub> = P<sub>robot</sub> + V[1:4]</li>
  <li>Filter: keep Z &gt; 0</li>
</ol>

</div>

In [6]:
def compute_point_cloud(M_v_rho, M_v_psi, M_v_pose_idx, H, pose_idx_min):
    """
    Generate a 3D point cloud from vertical sonar measurements.
    
    Parameters
    ----------
    M_v_rho       : (n,) distances from the sonar
    M_v_psi       : (n,) sonar rotation angles in degrees
    M_v_pose_idx  : (n,) global pose indices
    H             : (m, 7) local trajectory [x, y, z, qw, qx, qy, qz]
    pose_idx_min  : int, offset to convert global idx → local idx
    
    Returns
    -------
    point_cloud : (k, 3) array of [x, y, z] world-frame points (Z > 0 only)
    """
    points = []
    
    for i in range(len(M_v_rho)):
        # --- Look up robot pose ---
        local_idx = int(M_v_pose_idx[i] - pose_idx_min)
        
        # Safety clamp (in case of boundary issues)
        local_idx = max(0, min(local_idx, H.shape[0] - 1)) 
        
        CURRENT_X    = H[local_idx, :3]   # robot position [x, y, z]
        CURRENT_QUAT = H[local_idx, 3:]   # robot orientation [qw, qx, qy, qz]
        
        # --- Body-frame quaternion from sonar angle ---
        theta2rad =  M_v_psi[i] / 2
        CURRENT_IMPACT_QUAT_B = [cos(theta2rad), -sin(theta2rad), 0, 0]
        
        # Frame alignment rotations (same as 7_POINT_CLOUD.ipynb)
        Rot_Z_pi = [cos(pi / 2), 0, 0, sin(pi / 2)]
        CURRENT_IMPACT_QUAT_B = quatmultiply(CURRENT_IMPACT_QUAT_B, Rot_Z_pi)
        
        Rot_Y_pi = [cos(pi / 2), 0, sin(pi / 2), 0]
        CURRENT_IMPACT_QUAT_B = quatmultiply(CURRENT_IMPACT_QUAT_B, Rot_Y_pi)
        
        # --- Combined rotation: robot × body ---
        CURRENT_IMPACT_QUAT_0 = quatmultiply(CURRENT_QUAT, CURRENT_IMPACT_QUAT_B)
        
        # --- Rotate the distance vector ---
        Impact_Dist = np.array([0, 0, 0, M_v_rho[i]])
        Q_temp = quatmultiply(
            CURRENT_IMPACT_QUAT_0,
            quatmultiply(Impact_Dist, quatconj(CURRENT_IMPACT_QUAT_0))
        )
        
        # --- World-frame point ---
        IMPACT_TEMP = CURRENT_X + Q_temp[1:4]
        
        # Filter: keep only points with Z > 0
        if IMPACT_TEMP[2] > 0:
            points.append(IMPACT_TEMP)
    
    if len(points) > 0:
        return np.array(points)
    else:
        return np.empty((0, 3))

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 15px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h4 style="color: #5d4e37; text-align: center; margin-bottom: 15px; font-weight: bold;">Compute Point Clouds & Save All Scan Data</h4>

<p style="text-align: justify; margin: 10px 0; color: #4a4a4a;">
For each scan in <code>SCAN_DATA</code>, compute the 3D point cloud from the vertical sonar data, then flatten everything (including the <code>POINT_CLOUD</code>) into a single dict and save to <code>HDF5_DATA/7_SCANS.h5</code>.
</p>

<p style="text-align: justify; margin: 10px 0; color: #4a4a4a;">
The resulting HDF5 structure has scan groups <code>SCANS/0001</code>, <code>SCANS/0002</code>, ..., each containing: <code>M_v_rho</code>, <code>M_v_psi</code>, <code>M_v_pose_idx</code>, <code>M_v_time</code>, <code>M_h_M</code>, <code>M_h_pose_idx</code>, <code>M_h_time</code>, <code>H</code>, <code>s</code>, <code>scan_number</code>, <code>pose_idx_min</code>, <code>pose_idx_max</code>, and <code>POINT_CLOUD</code> (shape <code>(n_points, 3)</code>).
</p>

</div>

In [7]:
# ---------------------------------------------------------------------------
# Compute point clouds for each scan and build the flattened save structure
# ---------------------------------------------------------------------------
FLATTENED_SCAN_DATA = []
scan_point_clouds = {}   # scan_number → point cloud array (for later export)
total_points = 0

print(f"Processing {len(SCAN_DATA)} scans...\n")

for idx, scan in enumerate(SCAN_DATA):
    # --- Compute the point cloud for this scan ---
    pc = compute_point_cloud(
        M_v_rho      = scan['M_v']['rho'],
        M_v_psi      = scan['M_v']['psi'],
        M_v_pose_idx = scan['M_v']['pose_idx'],
        H            = scan['H'],
        pose_idx_min = scan['pose_range'][0]
    )
    scan_point_clouds[scan['scan_number']] = pc
    total_points += pc.shape[0]
    
    # Progress update every 50 scans
    if (idx + 1) % 50 == 0 or (idx + 1) == len(SCAN_DATA):
        print(f"  Processed {idx + 1}/{len(SCAN_DATA)} scans  |  Scan {scan['scan_number']:04d}: {pc.shape[0]} points")

    # --- Build the flattened dict for saving ---
    flat_scan = {
        # WRAP SCALARS IN BRACKETS TO MAKE THEM 1D ARRAYS!
        'scan_number':  [scan['scan_number']],
        
        # Flatten M_v
        'M_v_rho':      scan['M_v']['rho'],
        'M_v_psi':      scan['M_v']['psi'],
        'M_v_pose_idx': scan['M_v']['pose_idx'],
        'M_v_time':     scan['M_v']['time'],
        
        # Flatten M_h
        'M_h_M':        scan['M_h']['M'],
        'M_h_pose_idx': scan['M_h']['pose_idx'],
        'M_h_time':     scan['M_h']['time'],
        
        # Direct Arrays
        'H':            scan['H'],
        's':            scan['s'],
        
        # WRAP SCALARS IN BRACKETS!
        'pose_idx_min': [scan['pose_range'][0]],
        'pose_idx_max': [scan['pose_range'][1]],
        
        # Point cloud (n_points, 3) — integrated from 7_point_cloud.ipynb
        'POINT_CLOUD':  pc,
    }
    FLATTENED_SCAN_DATA.append(flat_scan)

print(f"\n✓ Done! Total valid points across all scans: {total_points}")

# --- Save everything to HDF5 ---
OUTPUT_FILE = 'HDF5_DATA/7_SCANS.h5'
save(OUTPUT_FILE, 'SCANS', FLATTENED_SCAN_DATA, mode='w')

Processing 303 scans...

  Processed 50/303 scans  |  Scan 0050: 161 points
  Processed 100/303 scans  |  Scan 0100: 197 points
  Processed 150/303 scans  |  Scan 0150: 167 points
  Processed 200/303 scans  |  Scan 0200: 166 points
  Processed 250/303 scans  |  Scan 0250: 175 points
  Processed 300/303 scans  |  Scan 0300: 212 points
  Processed 303/303 scans  |  Scan 0303: 44 points

✓ Done! Total valid points across all scans: 54372
Saved to HDF5_DATA/7_SCANS.h5 -> SCANS


<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 15px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h4 style="color: #5d4e37; text-align: center; margin-bottom: 15px; font-weight: bold;">Export Combined Point Cloud (PLY & PCD)</h4>

<p style="text-align: justify; margin: 10px 0; color: #4a4a4a;">
Combine all scan point clouds into one and export as <code>.ply</code> (via PyVista) and <code>.pcd</code> (via Open3D) for external visualization.
</p>

</div>

In [8]:
# Combine all point clouds
all_points = np.vstack([pc for pc in scan_point_clouds.values() if pc.shape[0] > 0])
print(f"Total points: {all_points.shape[0]}")

output_dir = "Point_cloud"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------------
# 1. Save as PLY using PyVista (includes elevation scalar)
# ---------------------------------------------------------
point_cloud = pv.PolyData(all_points)
point_cloud["elevation"] = all_points[:, 2]

ply_filename = os.path.join(output_dir, "fontaine_de_nimes_from_scans.ply")
point_cloud.save(ply_filename)
print(f"✓ PLY saved to: {ply_filename}")

# ---------------------------------------------------------
# 2. Save as PCD using Open3D
# ---------------------------------------------------------
try:
    import open3d as o3d
    pcd_filename = os.path.join(output_dir, "fontaine_de_nimes_from_scans.pcd")
    
    # Initialize Open3D point cloud object and assign points
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(all_points)
    
    # Save the file
    o3d.io.write_point_cloud(pcd_filename, pcd)
    print(f"✓ PCD saved to: {pcd_filename}")
    
except ImportError:
    print("\n⚠️ Note: Could not save .pcd. The 'open3d' library is required to save .pcd files.")
    print("You can install it by running: !pip install open3d in a new notebook cell.")

print(f"\nNumber of points processed: {len(all_points)}")

Total points: 54372
✓ PLY saved to: Point_cloud/fontaine_de_nimes_from_scans.ply
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
✓ PCD saved to: Point_cloud/fontaine_de_nimes_from_scans.pcd

Number of points processed: 54372


<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 15px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h4 style="color: #5d4e37; text-align: center; margin-bottom: 15px; font-weight: bold;">Quick Summary</h4>

<p style="text-align: justify; margin: 10px 0; color: #4a4a4a;">
Print a summary table showing the number of vertical sonar impacts vs. point cloud points per scan.
</p>

</div>

In [9]:
print(f"{'Scan':>6}  {'M_v impacts':>12}  {'Point Cloud':>12}")
print("-" * 34)

for scan, pc in list(zip(SCAN_DATA, scan_point_clouds.values()))[:10]:
    key = f"{scan['scan_number']:04d}"
    n_mv = len(scan['M_v']['rho'])
    n_pc = pc.shape[0]
    print(f"{key:>6}  {n_mv:>12}  {n_pc:>12}")

print(f"{'...':>6}")

# Last scan
last_scan = SCAN_DATA[-1]
last_pc = list(scan_point_clouds.values())[-1]
last_key = f"{last_scan['scan_number']:04d}"
print(f"{last_key:>6}  {len(last_scan['M_v']['rho']):>12}  {last_pc.shape[0]:>12}")

  Scan   M_v impacts   Point Cloud
----------------------------------
  0001           157           154
  0002           203           194
  0003           201           175
  0004           197           168
  0005           204           199
  0006           202           197
  0007           184           159
  0008           136           128
  0009           156           156
  0010           192           192
   ...
  0303            44            44


In [10]:
with h5py.File(OUTPUT_FILE, 'a') as f:
    for key, pc in scan_point_clouds.items():
        grp = f[f'SCANS/{key:04d}'] 
        
        # Remove old dataset if it exists
        if 'POINT_CLOUD' in grp:
            del grp['POINT_CLOUD']
        
        # Save with gzip compression
        grp.create_dataset('POINT_CLOUD', data=pc, compression='gzip', compression_opts=4)

print(f"✓ Saved POINT_CLOUD to all {len(scan_point_clouds)} scan groups in {OUTPUT_FILE}")

✓ Saved POINT_CLOUD to all 303 scan groups in HDF5_DATA/7_SCANS.h5


# 4.2.1. Environment Prior $f_p$

#### In this section, we explain how our environment prior $f_p$ is defined. Based on typical karstic environment, the natural choice is to use an elliptic cylinder as a prior shape. For a given pose $\boldsymbol{\eta}$ on the trajectory (equivalently its curvilinear abscissa) and a given sonar rotation angle $\psi^v$, $f_p(\boldsymbol{\eta}, \psi^v)$ returns the distance to the prior shape.

In order to fit this primitive (πρωτότυπο)  on the vertical sonar measurements, we follow an approach similar to [31].

* First we need a good initial shape. It is obtained by computing robust principal component analysis (RPCA) [32] on the vertical sonar point cloud. It provides three principals vectors which, in the case of a perfect elliptic cylinder, should correspond to the cylinder axis, the major and the minor elliptic axis. 

* To find the cylinder axis among those principal vectors, we used the fact that every tangent on a cylinder surface is orthogonal to its axis. This means that by approximating tangent vectors by the vector formed by two successive points, their projections on the PCA bases should be minimal along the cylinder axis.

* Then, the elliptic section is estimated by fitting the projections of all the points in the plane spanned by the two other principals axis [33]. The PCA base is then rotated around the cylinder axis to be aligned with the ellipse axis.

As can be seen with the orange elliptic cylinder in Figure 5, the dissymetric distribution of the vertical sonar points has a huge influence on the RPCA result. 

* To further improve our estimation, we use the Levendberg–Marquardt (LM) algorithm to obtain the shape which minimizes orthogonal distances to the elliptic cylinder. The estimation vector $x \in \mathbb{R}^8$ representing our elliptic cylinder is composed of a 6D pose in $SE(3)$ and the scales along the minor and major ellipse axis. Here, we use on-manifold optimization on $SE(3)$, as described in [22]. The final prior can be seen as the purple cylinder in Figure 5.

  First we need a good initial shape. It is obtained by computing robust principal component analysis (RPCA) [32] on the vertical sonar point cloud. It provides three principals vectors which, in the case of a perfect elliptic cylinder, should correspond to the cylinder axis, the major and the minor elliptic axis. 

In [ ]:
# Import the convenience function from your local rpca.py file
# ---------------------------------------------------------------------------
# Run the Trimmed Grassmann Average (TGA) RPCA
# ---------------------------------------------------------------------------
print("Starting TGA-RPCA on point clouds...\n")

# This will process all scans in the file, extracting K=3 components per scan.
# It automatically skips any scans with fewer than 3 points.
rpca_results = rpca_on_scans(
    scan_ids=3, 
    h5_path='HDF5_DATA/7_SCANS.h5', 
    K=3, 
    trim_pct=0.50, 
    verbose=True
)

# ---------------------------------------------------------------------------
# Inspect the 3 principal vectors for the very first valid scan
# ---------------------------------------------------------------------------
if rpca_results:
    first_scan_id = list(rpca_results.keys())[0]
    first_scan_data = rpca_results[first_scan_id]
    
    print(f"\n--- Summary for Scan {first_scan_id:04d} ---")
    print(f"Number of points used: {first_scan_data['n_points']}")
    print(f"Robust Center (Median): \n{first_scan_data['center']}")
    print(f"Captured Variances: \n{first_scan_data['variances']}")
    print("The 3 Principal Vectors (Components Matrix):")
    print(first_scan_data['components'])

Starting TGA-RPCA on point clouds...

  Scan 0003  |   175 pts  |  variances = [57.04631546 36.12291876 17.06082586]

✓ Processed 1 scan(s)

--- Summary for Scan 0003 ---
Number of points used: 175
Robust Center (Median): 
[-0.78189396  0.83403747  0.7878104 ]
Captured Variances: 
[57.04631546 36.12291876 17.06082586]
The 3 Principal Vectors (Components Matrix):
[[ 0.2091504  -0.84659211 -0.48942611]
 [ 0.32767419  0.52740622 -0.78388284]
 [ 0.92470788  0.0021103   0.38067163]]


To find the cylinder axis among those principal vectors, we used the fact that every tangent on a cylinder surface is orthogonal to its axis. This means that by approximating tangent vectors by the vector formed by two successive points, their projections on the PCA bases should be minimal along the cylinder axis.

In [ ]:
def find_cylinder_axis(point_cloud, components):
    """
    Identify the cylinder axis among 3 principal vectors from RPCA.
    Section 4.2.1: tangent projections are minimal along the axis.
    """
    tangents = np.diff(point_cloud, axis=0)
    projection_sums = np.zeros(3)
    for k in range(3):
        projection_sums[k] = np.sum(np.abs(tangents @ components[k]))

    axis_index  = np.argmin(projection_sums)
    axis_vector = components[axis_index]
    return axis_index, axis_vector, projection_sums

def find_cylinder_axes_on_scans(scan_ids='all',
                                h5_path='HDF5_DATA/7_SCANS.h5',
                                K=3, trim_pct=0.50, seed=42,
                                verbose=True):
    """
    Run RPCA then identify the cylinder axis for each requested scan.

    Parameters
    ----------
    scan_ids  : int, list[int], or 'all'
    h5_path   : str
    K, trim_pct, seed : forwarded to rpca_on_scans
    verbose   : bool

    Returns
    -------
    results : dict { scan_number: { axis_vector, major_axis, minor_axis, ... } }
    """
    rpca_results = rpca_on_scans(
        scan_ids=scan_ids, h5_path=h5_path,
        K=K, trim_pct=trim_pct, seed=seed, verbose=verbose,
    )

    results = {}
    with h5py.File(h5_path, 'r') as f:
        for scan_num, rpca_data in rpca_results.items():
            key = f'{scan_num:04d}'
            pc  = f[f'SCANS/{key}/POINT_CLOUD'][:]

            axis_idx, axis_vec, proj_sums = find_cylinder_axis(
                pc, rpca_data['components']
            )

            other_idx = sorted([i for i in range(K) if i != axis_idx])
            # The two remaining vectors span the ellipse cross-section.
            # Order them by decreasing variance → major then minor.
            vr = rpca_data['variances']
            if vr[other_idx[0]] < vr[other_idx[1]]:
                other_idx = [other_idx[1], other_idx[0]]

            major_axis = rpca_data['components'][other_idx[0]]
            minor_axis = rpca_data['components'][other_idx[1]]

            results[scan_num] = {
                'axis_index':      axis_idx,
                'axis_vector':     axis_vec,
                'major_axis':      major_axis,
                'minor_axis':      minor_axis,
                'components':      rpca_data['components'],
                'center':          rpca_data['center'],
                'variances':       rpca_data['variances'],
                'projection_sums': proj_sums,
                'n_points':        rpca_data['n_points'],
            }

            if verbose:
                print(f"\n--- Scan {key} ({rpca_data['n_points']} pts) ---")
                print(f"  Projection sums : {proj_sums}")
                print(f"  Cylinder axis   : {axis_vec}  (component #{axis_idx})")
                print(f"  Major ellipse   : {major_axis}  (component #{other_idx[0]})")
                print(f"  Minor ellipse   : {minor_axis}  (component #{other_idx[1]})")

    if verbose:
        print(f"\n✓ Identified cylinder axis for {len(results)} scan(s)")

    return results


In [13]:
results = find_cylinder_axes_on_scans(scan_ids=3)

  Scan 0003  |   175 pts  |  variances = [57.04631546 36.12291876 17.06082586]

✓ Processed 1 scan(s)

--- Scan 0003 (175 pts) ---
  Projection sums : [57.4892315  31.60509517  9.91033011]
  Cylinder axis   : [0.92470788 0.0021103  0.38067163]  (component #2)
  Major ellipse   : [ 0.2091504  -0.84659211 -0.48942611]  (component #0)
  Minor ellipse   : [ 0.32767419  0.52740622 -0.78388284]  (component #1)

✓ Identified cylinder axis for 1 scan(s)


### I have to verify that:

One thing to be aware of
The ordering of points in point_cloud matters. Your tangents are only meaningful if consecutive rows in the array are spatially adjacent — i.e., the points are ordered by sonar scan angle. Since you build the point cloud by iterating over M_v_psi in the order the sonar swept, the points are naturally angle-ordered within each scan revolution, which is exactly what you want. This assumption holds given how compute_point_cloud is structured.

Then, the elliptic section is estimated by fitting the projections of all the points in the plane spanned by the two other principals axis [33]. The PCA base is then rotated around the cylinder axis to be aligned with the ellipse axis.

In [14]:
"""
=======================
Implements the elliptic cross-section fitting step described in Section 4.2.1
of Breux & Lapierre (2020), using the Direct Least Squares ellipse fitting
method of Fitzgibbon, Pilu & Fisher (1999) — paper [33].

The ONLY public function is:

    fit_elliptic_sections(scan_pca_results)

It takes as input the dict produced by `find_cylinder_axes_on_scans`
(which already contains, per scan, the two non-axis PCA vectors and the
point cloud) and returns, per scan, the fitted elliptic section.

References
----------
  [33] A. Fitzgibbon, M. Pilu, R. B. Fisher,
       "Direct Least Square Fitting of Ellipses,"
       IEEE Trans. PAMI, vol. 21, no. 5, pp. 476–480, May 1999.
"""

# ---------------------------------------------------------------------------
# fit_elliptic_sections  —  the single public function
# ---------------------------------------------------------------------------

def fit_elliptic_sections(scan_pca_results):
    """
    Estimate the elliptic cross-section for every scan's point cloud.

    This is the direct Python translation of the paper's pipeline
    (Section 4.2.1 + Fitzgibbon et al. 1999, Fig. 4):

      For each scan:
        1. Centre the 3-D point cloud at the robust RPCA centroid.
        2. Project all points onto the 2-D plane spanned by the two
           non-axis PCA vectors  (major_axis, minor_axis).
        3. Build the N×6 design matrix   D  with rows [u², uv, v², u, v, 1].
        4. Build the 6×6 scatter matrix  S = Dᵀ D.
        5. Build the 6×6 ellipse-specific constraint matrix C
           (encodes  4ac − b² = 1,  Eq. 4 of [33]).
        6. Solve the generalised eigensystem  S·a = λ·C·a   (Eq. 7).
        7. Select the eigenvector whose generalised eigenvalue is the
           ONLY NEGATIVE finite one  — exactly as the 6-line Matlab code
           in [33] does with  find(geval < 0 & ~isinf(geval)).
        8. Extract the ellipse orientation angle θ in the 2-D PCA plane.
        9. Rotate (major_axis, minor_axis) around the cylinder axis by θ
           so that the PCA base aligns with the true ellipse axes
           (last sentence of Section 4.2.1).

    Parameters
    ----------
    scan_pca_results : dict  { scan_number (int) : pca_dict }
        Output of `find_cylinder_axes_on_scans`.  Each pca_dict must contain:
            'point_cloud'  : (N, 3)  3-D world-frame points for this scan
            'center'       : (3,)    robust RPCA centroid
            'major_axis'   : (3,)    first non-cylinder PCA unit vector
            'minor_axis'   : (3,)    second non-cylinder PCA unit vector
            'axis_vector'  : (3,)    cylinder axis unit vector

    Returns
    -------
    results : dict  { scan_number (int) : result_dict }
        Each result_dict contains:
            'a'          : (6,)   algebraic coefficients [a,b,c,d,e,f]
            'center_2d'  : (2,)   ellipse centre in the PCA plane  (u₀, v₀)
            'semi_axes'  : (2,)   (semi_major, semi_minor)  lengths
            'angle_rad'  : float  ellipse orientation angle in the PCA plane
            'new_major'  : (3,)   major-axis 3-D unit vector (ellipse-aligned)
            'new_minor'  : (3,)   minor-axis 3-D unit vector (ellipse-aligned)
            'u'          : (N,)   projected u-coordinates  (diagnostic)
            'v'          : (N,)   projected v-coordinates  (diagnostic)
    """
    results = {}

    for scan_num, pca in scan_pca_results.items():

        point_cloud   = pca['point_cloud']   # (N, 3)
        center        = pca['center']        # (3,)
        major_axis    = pca['major_axis']    # (3,)
        minor_axis    = pca['minor_axis']    # (3,)
        cylinder_axis = pca['axis_vector']   # (3,)

        N = point_cloud.shape[0]
        if N < 6:
            # Cannot fit an ellipse with fewer than 6 points
            continue

        # ------------------------------------------------------------------
        # Step 1 — centre the point cloud at the RPCA centroid
        # ------------------------------------------------------------------
        pts = point_cloud - center           # (N, 3)

        # ------------------------------------------------------------------
        # Step 2 — project onto the 2-D plane spanned by (major, minor)
        #   u_i = (p_i − center) · major_axis
        #   v_i = (p_i − center) · minor_axis
        # ------------------------------------------------------------------
        u = pts @ major_axis                 # (N,)
        v = pts @ minor_axis                 # (N,)

        # ------------------------------------------------------------------
        # Step 3 — design matrix  D  (paper Fig. 4, first line)
        #   Each row:  [u², u·v, v², u, v, 1]
        # ------------------------------------------------------------------
        D = np.column_stack([
            u * u,          # u²
            u * v,          # u·v
            v * v,          # v²
            u,              # u
            v,              # v
            np.ones(N),     # 1
        ])                  # shape (N, 6)

        # ------------------------------------------------------------------
        # Step 4 — scatter matrix  S = Dᵀ D   (paper Fig. 4, third line)
        # ------------------------------------------------------------------
        S = D.T @ D         # (6, 6)

        # ------------------------------------------------------------------
        # Step 5 — constraint matrix  C   (paper Eq. 4 and Fig. 4)
        #
        #   Constraint:  4ac − b² = 1   ⟺   aᵀ C a = 1
        #
        #   With a = [a, b, c, d, e, f]  (indices 0..5):
        #       aᵀ C a = 4·a[0]·a[2] − a[1]²
        #   =>  C[0,2] = C[2,0] = 2,   C[1,1] = −1,   rest = 0.
        #
        #   In the Matlab listing:
        #       C(6,6)=0; C(1,3)=-2; C(2,2)=1; C(3,1)=-2;
        #   Note: Matlab is 1-indexed and the sign convention there
        #   encodes −(4ac − b²) = 1  (i.e. b² − 4ac = 1), which is why
        #   the Matlab code looks for the *negative* eigenvalue.
        #   Here we use the sign from Eq. 4 of the paper (4ac − b² = 1),
        #   which makes the ellipse solution the *positive* eigenvalue in
        #   scipy — BUT, to stay byte-for-byte faithful to the 6-line
        #   Matlab code, we adopt the SAME Matlab sign (b² − 4ac = 1) so
        #   that we can search for the negative eigenvalue just like Matlab.
        # ------------------------------------------------------------------
        C = np.zeros((6, 6))
        C[0, 2] = -2.0    # Matlab: C(1,3) = -2  (1-indexed → 0-indexed)
        C[2, 0] = -2.0    # Matlab: C(3,1) = -2
        C[1, 1] =  1.0    # Matlab: C(2,2) =  1

        # ------------------------------------------------------------------
        # Step 6 — solve  S·a = λ·C·a   (paper Eq. 7 and Matlab Fig. 4)
        #
        #   scipy.linalg.eig(A, B) solves  A·v = λ·B·v,
        #   i.e.  S·a = λ·C·a  — identical to Matlab's eig(S, C).
        # ------------------------------------------------------------------
        eigenvalues, eigenvectors = eig(S, C)

        # ------------------------------------------------------------------
        # Step 7 — select the unique elliptical solution
        #
        #   The 6-line Matlab code (paper Fig. 4):
        #       [NegR, NegC] = find(geval < 0 & ~isinf(geval));
        #       a = gevec(:, NegC);
        #
        #   With the Matlab sign convention (b² − 4ac = 1) adopted above,
        #   the ellipse solution is the ONLY NEGATIVE finite eigenvalue.
        # ------------------------------------------------------------------
        real_mask = np.isreal(eigenvalues) & np.isfinite(eigenvalues)
        ev_real   = eigenvalues[real_mask].real
        eg_real   = eigenvectors[:, real_mask].real

        neg_mask  = ev_real < 0
        if not np.any(neg_mask):
            # Degenerate point set — skip this scan
            continue

        # Among (potentially multiple) negative eigenvalues pick the one
        # with the largest magnitude — this is the unique "most elliptic"
        # solution and mirrors Matlab's behaviour when only one is present.
        neg_indices = np.where(neg_mask)[0]
        best_idx    = neg_indices[np.argmin(ev_real[neg_indices])]
        a           = eg_real[:, best_idx]   # (6,)

        # ------------------------------------------------------------------
        # Step 8 — extract the ellipse orientation angle θ in the PCA plane
        #
        #   Standard conic-to-geometric conversion:
        #       θ = ½ · arctan2(b, a − c)
        #   where a = a[0], b = a[1], c = a[2].
        #
        #   Then decide which axis is the major one using the eigenvalues of
        #   the 2×2 shape matrix  [[a, b/2],[b/2, c]].
        # ------------------------------------------------------------------
        a0, b0, c0, d0, e0, f0 = a

        # centre of the fitted ellipse in the 2-D PCA plane
        denom    = b0**2 - 4.0 * a0 * c0       # < 0 for an ellipse
        if abs(denom) < 1e-14:
            continue                            # numerical degenerate
        u0_ell   = (2.0 * c0 * d0 - b0 * e0) / denom
        v0_ell   = (2.0 * a0 * e0 - b0 * d0) / denom

        # semi-axis lengths
        disc     = np.sqrt(max((a0 - c0)**2 + b0**2, 0.0))
        num      = 2.0 * (a0 * e0**2 + c0 * d0**2 - b0 * d0 * e0
                          + denom * f0)
        val1     = num * ((a0 + c0) + disc)
        val2     = num * ((a0 + c0) - disc)
        if val1 < 0 or val2 < 0:
            continue                            # not a real ellipse
        semi1    = abs(np.sqrt(val1) / denom)
        semi2    = abs(np.sqrt(val2) / denom)
        semi_major = max(semi1, semi2)
        semi_minor = min(semi1, semi2)

        # orientation: θ = ½ arctan2(b, a − c)
        if abs(a0 - c0) < 1e-12:
            theta = np.pi / 4.0 if b0 > 0 else -np.pi / 4.0
        else:
            theta = 0.5 * np.arctan2(b0, a0 - c0)

        # eigenvalues of the 2×2 shape matrix to decide which axis is major
        lam1 = 0.5 * ((a0 + c0) + disc)
        lam2 = 0.5 * ((a0 + c0) - disc)
        if abs(lam1) > abs(lam2):
            theta = theta + np.pi / 2.0

        # wrap into (−π/2, π/2]
        theta = (theta + np.pi / 2.0) % np.pi - np.pi / 2.0

        # ------------------------------------------------------------------
        # Step 9 — rotate PCA basis around the cylinder axis by θ
        #
        #   "The PCA base is then rotated around the cylinder axis to be
        #    aligned with the ellipse axis."  (Section 4.2.1)
        #
        #   Rodrigues' rotation formula:
        #       R(θ, ax)·v = cos θ·v  + sin θ·(ax × v)  + (1−cos θ)(ax·v)·ax
        # ------------------------------------------------------------------
        ax    = cylinder_axis / np.linalg.norm(cylinder_axis)
        cos_t = np.cos(theta)
        sin_t = np.sin(theta)

        def _rodrigues(vec):
            return (cos_t * vec
                    + sin_t * np.cross(ax, vec)
                    + (1.0 - cos_t) * np.dot(ax, vec) * ax)

        new_major = _rodrigues(major_axis)
        new_minor = _rodrigues(minor_axis)
        new_major /= np.linalg.norm(new_major)
        new_minor /= np.linalg.norm(new_minor)

        # ------------------------------------------------------------------
        # Store result for this scan
        # ------------------------------------------------------------------
        results[scan_num] = {
            'a'         : a,
            'center_2d' : np.array([u0_ell, v0_ell]),
            'semi_axes' : np.array([semi_major, semi_minor]),
            'angle_rad' : theta,
            'new_major' : new_major,
            'new_minor' : new_minor,
            'u'         : u,
            'v'         : v,
        }

    return results

In [15]:
pca_results = results # Grab the PCA results you already computed in Cell 13  

with h5py.File('HDF5_DATA/7_SCANS.h5', 'r') as f: # Manually inject the missing 'point_cloud' data into the dictionary
    for scan_num in pca_results.keys():
        key = f'{scan_num:04d}'
        pca_results[scan_num]['point_cloud'] = f[f'SCANS/{key}/POINT_CLOUD'][:] # Add the point cloud so fit_elliptic_sections can use it

ellipse_results = fit_elliptic_sections(pca_results) # Run the ellipse fitting function


# ----------------------- Print ------------------------------------#

if not ellipse_results:
    print("No valid ellipses could be fitted (points might be degenerate).")
else:
    for scan_num, ell in ellipse_results.items():
        print(f"--- Ellipse Fit Results for Scan {scan_num:04d} ---")
        print(f"  Center (2D PCA plane) : ({ell['center_2d'][0]:+.4f}, {ell['center_2d'][1]:+.4f})")
        print(f"  Semi-axes             : major={ell['semi_axes'][0]:.4f}, minor={ell['semi_axes'][1]:.4f}")
        print(f"  Orientation Angle     : {np.degrees(ell['angle_rad']):.2f}°")
        print(f"  Rotated Major Axis    : {ell['new_major']}")
        print(f"  Rotated Minor Axis    : {ell['new_minor']}")
        print("-" * 45)

--- Ellipse Fit Results for Scan 0003 ---
  Center (2D PCA plane) : (+0.4929, -0.6861)
  Semi-axes             : major=1.9240, minor=0.6886
  Orientation Angle     : -16.23°
  Rotated Major Axis    : [ 0.11122209 -0.96160062 -0.25090616]
  Rotated Minor Axis    : [ 0.37140315  0.26892042 -0.88867402]
---------------------------------------------
